# 05 — Pre-Match Prediction

Predict the winner of a CS match before it starts — using only the two team names,
the match format, and optionally the stage.

**No current-match data used.** All features come from each team's historical record.

Run `03_training.ipynb` first to generate the saved model.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from conf.settings import PROCESSED_DIR, DATASET_FILE
from src.module.model.model import CSWinnerModel
from src.module.model.explain import ModelExplainer
from src.module.model.features import build_features, build_team_stats_lookup

## 1. Load Saved Model

In [ ]:
model = CSWinnerModel.load()
print('Model loaded.')

## 2. Build Team Stats Lookup

`build_team_stats_lookup(df)` processes the full feature DataFrame once and returns
a dict `{team_name: {stat_name: value}}` with each team's most recent pre-match state.

Build it once per session and reuse — it is the historical context for every prediction.

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / DATASET_FILE)
df = build_features(df)

lookup = build_team_stats_lookup(df)
teams = sorted(lookup.keys())

print(f'Teams in lookup: {len(teams)}')
print(f'Stats per team:  {len(next(iter(lookup.values())))}')

## 3. Find Teams

Search by partial name to find the correct spelling used in the dataset.

In [ ]:
query = 'navi'  # change this to search
matches = [t for t in teams if query.lower() in t.lower()]
print(f'Teams matching "{query}":', matches)

## 4. Predict a Match

Change `TEAM1`, `TEAM2`, `FORMAT` and `STAGE` to predict any matchup.

In [ ]:
TEAM1  = teams[0]   # replace with e.g. 'Natus Vincere'
TEAM2  = teams[1]   # replace with e.g. 'Astralis'
FORMAT = 'bo3'      # 'bo1', 'bo3' or 'bo5'
STAGE  = 'Grand Final'  # optional; affects pressure_score feature

result = model.predict_match(
    team1=TEAM1,
    team2=TEAM2,
    match_format=FORMAT,
    stage=STAGE,
    team_stats=lookup,
)

print()
print('=' * 45)
print(f"  {result['team1']}  vs  {result['team2']}")
print(f"  Format: {FORMAT}   Stage: {STAGE}")
print('=' * 45)
print(f"  Predicted winner : {result['predicted_winner']}")
print(f"  P(team1 wins)    : {result['prob_team1_wins']:.1%}")
print(f"  P(team2 wins)    : {result['prob_team2_wins']:.1%}")
print(f"  ELO difference   : {result['elo_diff']:+.1f}")
print('=' * 45)

## 5. Explain the Prediction (SHAP Waterfall)

Shows which features pushed the model toward team1 or team2 for this specific matchup.

In [ ]:
# Build the raw feature row the same way predict_match() does internally
row = {}
for stat, val in lookup.get(TEAM1, {}).items():
    row[f'team1_{stat}'] = val
for stat, val in lookup.get(TEAM2, {}).items():
    row[f'team2_{stat}'] = val
row['format_maps'] = {'bo1': 1, 'bo3': 3, 'bo5': 5}.get(FORMAT.lower(), 3)

explainer = ModelExplainer(model=model)
explainer.explain_match(row, save=False)

## 6. Batch Predict Multiple Matchups

In [ ]:
matchups = [
    (teams[0], teams[1], 'bo3'),
    (teams[2], teams[3], 'bo1'),
    (teams[4], teams[5], 'bo5'),
]

rows = []
for t1, t2, fmt in matchups:
    r = model.predict_match(t1, t2, match_format=fmt, team_stats=lookup)
    rows.append({
        'team1': r['team1'],
        'team2': r['team2'],
        'format': fmt,
        'predicted_winner': r['predicted_winner'],
        'p_team1': r['prob_team1_wins'],
        'p_team2': r['prob_team2_wins'],
        'elo_diff': r['elo_diff'],
    })

pd.DataFrame(rows).style.format({'p_team1': '{:.1%}', 'p_team2': '{:.1%}', 'elo_diff': '{:+.1f}'})

## 7. ELO Leaderboard — Top 30 Teams

In [ ]:
elo_rows = [
    {
        'team': team,
        'elo': stats.get('elo', 1000.0),
        'overall_wr': stats.get('overall_wr', float('nan')),
        'total_matches': int(stats.get('total_matches', 0)),
        'win_streak': int(stats.get('win_streak_all', 0)),
    }
    for team, stats in lookup.items()
]
elo_df = (
    pd.DataFrame(elo_rows)
    .sort_values('elo', ascending=False)
    .reset_index(drop=True)
)
elo_df.index += 1
elo_df.head(30).style.format({
    'elo': '{:.1f}',
    'overall_wr': '{:.1%}',
})